In [3]:
!pip install xpress

  Using cached xpress-9.4.3-cp312-cp312-manylinux1_x86_64.whl.metadata (5.1 kB)
Using cached xpress-9.4.3-cp312-cp312-manylinux1_x86_64.whl (2.1 MB)


In [2]:
!pip uninstall xpress -y

Found existing installation: xpress 9.4.3
Uninstalling xpress-9.4.3:
  Successfully uninstalled xpress-9.4.3


In [1]:
import xpress as xp
xp.init("/workspaces/coatwork-vrp-challenge/xpauth.xpr")

In [2]:
from read_data import *

In [3]:
couriers = load_couriers_from_csv("/workspaces/coatwork-vrp-challenge/couriers (1).csv")
deliveries = load_deliveries_from_csv("/workspaces/coatwork-vrp-challenge/deliveries (1).csv")
distance_matrix = load_travel_time_from_csv("/workspaces/coatwork-vrp-challenge/traveltimes (1).csv")
distance_matrix = [row[1:] for row in distance_matrix[1:]]

In [4]:
D = [delivery.dropoff_loc for delivery in deliveries]
D

[8, 8, 11, 13, 8]

In [7]:
distance_matrix

[['Locations',
  '1',
  '2',
  '3',
  '4',
  '5',
  '6',
  '7',
  '8',
  '9',
  '10',
  '11',
  '12',
  '13'],
 [1, 0, 5, 11, 5, 4, 4, 8, 10, 5, 8, 8, 5, 6],
 [2, 5, 0, 6, 0, 1, 1, 4, 5, 0, 3, 11, 0, 3],
 [3, 11, 6, 0, 7, 7, 7, 3, 2, 6, 3, 17, 6, 5],
 [4, 5, 0, 7, 0, 1, 1, 4, 5, 1, 4, 11, 0, 3],
 [5, 4, 1, 7, 1, 0, 1, 4, 5, 1, 4, 10, 1, 4],
 [6, 4, 1, 7, 1, 1, 0, 4, 5, 1, 4, 10, 1, 4],
 [7, 8, 4, 3, 4, 4, 4, 0, 4, 4, 0, 14, 4, 2],
 [8, 10, 5, 2, 5, 5, 5, 4, 0, 5, 4, 15, 5, 4],
 [9, 5, 0, 6, 1, 1, 1, 4, 5, 0, 4, 12, 1, 3],
 [10, 8, 3, 3, 4, 4, 4, 0, 4, 4, 0, 14, 4, 2],
 [11, 8, 11, 17, 11, 10, 10, 14, 15, 12, 14, 0, 10, 12],
 [12, 5, 0, 6, 0, 1, 1, 4, 5, 1, 4, 10, 0, 3],
 [13, 6, 3, 5, 3, 4, 4, 2, 4, 3, 2, 12, 3, 0]]

In [4]:
def initialize_problem_inputs(couriers, deliveries, distance_matrix):
    """
    Generates the vertices, arcs, and relevant input data based on the couriers and deliveries.
    :param couriers: List of Courier objects
    :param deliveries: List of Delivery objects
    :param distance_matrix: Matrix of distances between locations
    :return: V (vertices), A (arcs), K (vehicles), T_uv (travel times), l_v (time window lower bound),
             u_v (time window upper bound), Q_v (pickup quantities), Q_max (vehicle capacities), Q_sk (start quantities)
    """
    # Vertices: Locations of couriers, pickup, and dropoff points
    p = [delivery.pickup_loc-1 for delivery in deliveries]
    d = [delivery.dropoff_loc-1 for delivery in deliveries]
    s = [courier.location-1 for courier in couriers]
    
    # Combine all unique locations into a set of vertices
    V = list(range(len(distance_matrix)))
    
    

    # Arcs: All possible pairs of distinct vertices
    A = [(i, j) for i in V for j in V]

    # Vehicles (K): The list of courier IDs
    K = [courier.courier_id-1 for courier in couriers]

    # Travel times (T_uv): Derived from the distance matrix
    T_uv = {(i, j): distance_matrix[i][j] for i in range(len(V)) for j in range(len(V))}
        

    # Time windows (l_v and u_v) and pickup quantities (Q_v)
    l_v = {delivery.pickup_loc: delivery.time_window_start for delivery in deliveries}
    u_v = {delivery.pickup_loc: delivery.time_window_start + 10000 for delivery in deliveries}  #
    Q_v = [0]*len(V)
    l_v = [0]*len(V)
    u_v = [0]*len(V)
    for delivery in deliveries:
        Q_v[delivery.pickup_loc-1] = delivery.capacity
        Q_v[delivery.dropoff_loc-1] = -delivery.capacity
        l_v[delivery.pickup_loc-1] = delivery.time_window_start
        u_v[delivery.dropoff_loc-1] = delivery.time_window_start + 10000
        
    #Q_v = [delivery.capacity for delivery in deliveries]

    # Vehicle capacities (Q_max) and start node capacities (Q_sk)
    Q_max = [courier.capacity for courier in couriers]
    Q_sk = [courier.capacity for courier in couriers]

    return V, A, K, T_uv, l_v, u_v, Q_v, Q_max, Q_sk, s


def build_model(deliveries, couriers, V, A, K, T_uv, l_v, u_v, Q_v, Q_max, Q_sk, s):
    """
    Builds the MIP model by adding the objective function and constraints.
    """

    x_uvk = problem.addVariables(len(V),len(V),len(K), vartype=xp.binary)
    t_v = problem.addVariables(len(V), name ="t")
    q_v = problem.addVariables(len(V), name ="q")

    # Objective: minimize sum of t_v (time at destinations)
    D = [delivery.dropoff_loc-1 for delivery in deliveries]
    problem.addObjective(xp.Sum(t_v[v] for v in D))

    # Constraints
    # (2) Each delivery must be visited once by one vehicle
    for v in V:
        problem.addConstraint(xp.Sum(x_uvk[u, v, k] for (u, v) in A for k in K) == 1)

    # (3) Flow conservation for each vertex and vehicle
    for v in V:
        for k in K:
            problem.addConstraint(
                xp.Sum(x_uvk[v, u, k] for (v, u) in A) == xp.Sum(x_uvk[u, v, k] for (u, v) in A)
            )

    # (4) Flow balance at pickups and deliveries
    for delivery in deliveries:
        o = delivery.pickup_loc-1
        d = delivery.dropoff_loc-1
        for k in K:
            problem.addConstraint(
                xp.Sum(x_uvk[u, o, k] for (u, o) in A) == xp.Sum(x_uvk[u, d, k] for (u, d) in A)
            )

    # (5) Each vehicle starts at its start node
    for k in K:
        start_node = s[k]
        problem.addConstraint(
            xp.Sum(x_uvk[s, v, k] for v in V for (s, v) in A) == 1
        )

    # (6) Time window constraints for each arc and vehicle
    for (u, v) in A:
        for k in K:
            problem.addConstraint(
                t_v[u] + (T_uv[(u, v)] + 10000) * x_uvk[u, v, k] <= t_v[v] + 10000
            )

    # (7) Pickup time constraints
    for delivery in deliveries:
        o = delivery.pickup_loc-1
        d = delivery.dropoff_loc-1
        problem.addConstraint(t_v[d]-t_v[o] >= T_uv[(o,d)])

    # (8) Capacity constraints for each arc and vehicle
    for (u, v) in A:
        for k in K:
            problem.addConstraint(
                    q_v[u] + (Q_v[v] + Q_max[k]) * x_uvk[u, v, k] <= q_v[v] + Q_max[k]
            )   

    # (9) Start node quantity constraint
    for k in K:
        problem.addConstraint(q_v[s] == Q_sk[k])

    # (10) Time window constraints for each vertex
    for v in V:
        problem.addConstraint(l_v[v] <= t_v[v] <= u_v[v])

    # (11) Quantity bounds for each vertex
    for v in V:
        problem.addConstraint(0 <= q_v[v] <= Q_max[0])

    return problem


def solve_model(problem, time_limit=30, heuristic_level=3):
    # Set the time limit and heuristic strategy level
    #problem.controls.maxtime = time_limit
    #problem.controls.heurstrategy = heuristic_level
    problem.controls.timelimit=time_limit
    problem.controls.heuremphasis=2

    problem.solve()
    return problem


In [5]:
problem = xp.problem()
V, A, K, T_uv, l_v, u_v, Q_v, Q_max, Q_sk, s = initialize_problem_inputs(couriers, deliveries, distance_matrix)


In [6]:
build_model(deliveries, couriers, V, A, K, T_uv, l_v, u_v, Q_v, Q_max, Q_sk,s)


In [7]:
solve_model(problem)

FICO Xpress v9.4.3, Hyper64, solve started 12:34:43, Sep 25, 2024
Heap usage: 1931KB (peak 1931KB, 349KB system)
Minimizing MILP noname using up to 2 threads and up to 7938MB memory, with these control settings:
OUTPUTLOG = 1
HEUREMPHASIS = 2
TIMELIMIT = 30
NLPPOSTSOLVE = 1
XSLP_DELETIONCONTROL = 0
XSLP_OBJSENSE = 1
Original problem has:
      2222 rows         1040 cols        20040 elements      1014 entities
 
 
The problem is infeasible due to row R7
Presolve finished in 0 seconds
Heap usage: 1526KB (peak 3395KB, 366KB system)
 *** Search completed ***
Problem is integer infeasible
  Solution time / primaldual integral :      0.00s/ 100.000000%
  Number of solutions found / nodes   :         0 /         0
